# 02 - Reading the index columns

**Purpose.** To understand `results/frame_index.csv` - specifically its *measured* columns, the
ones `01` computed from pixels rather than copied from a header. It takes **one frame of each
measured type**, shows the pixels those numbers came from, and says what each column is doing
and why the archive is worth keeping it in.

**What it is not.** Not a measurement. Nothing here is new evidence about the sensor, nothing is
written to `results/`, and no threshold is re-tuned. Where a number below looks like a sensor
property it is a property of *the estimator*, and the notebook says so. It is a reading aid for
`01` and for `astropix/stats.py` (`DECISIONS` D37 - a notebook exists for an agreed purpose).

**One frame of each type, held as still as possible.** All four exemplars are gain 252, setpoint
-10 C, so that when two columns differ the difference is the *frame type*, not the capture
settings. The archive is a test corpus (D36) and this is the one place it is being used exactly
as intended: real pixels, to make code legible.

In [ ]:
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropix import fits as F, spatial as S, stats as ST

pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "font.size": 8})

RESULTS = pathlib.Path("..") / "results"
INDEX = RESULTS / "frame_index.csv"

# The columns `01` measured from pixels.  Everything else in the index is either
# identity (path, size, mtime) or a header value copied verbatim.
#
# Of these, `classify` reads exactly one -- `level` (D50).  The rest are
# description and integrity: they make the corpus queryable and they catch a
# file that is not what it claims, and no branch of the classifier consults
# them.  `most_typical` below uses the whole set, because "typical" is a
# question about the frame, not about the verdict.
MEASURED = ["level", "sigma", "block_spread", "mult16_frac", "sat_frac"]
PER_PLANE = [f"med_{p.lower()}" for p in S.PLANES]
# The whole-frame summary `01` records alongside them.  Deliberately NOT in
# MEASURED: it is orientation, and `most_typical` should not be steered by it.
SUMMARY = ["mean", "median", "min", "max", "std", "sampled_px"]

idx = pd.read_csv(INDEX)
ok = idx[idx.status == "ok"]
print(f"{len(ok)} readable rows, {len(idx.columns)} columns")
print(f"{len(MEASURED) + len(PER_PLANE) + len(SUMMARY)} of them measured from pixels")

## Choosing the four frames

Picking "a typical bias" by eye is how a story gets told about an outlier. Instead: inside each
pool, take the frame whose measured columns are **closest to that pool's own median**, in units
of the pool's own spread. It is a one-line selection, it is reproducible, and it cannot be
talked into choosing something interesting.

The pools are deliberately narrow - one gain, one setpoint, one exposure per type, and the
lights restricted to three nights of the NGC 7000 ladder (`results/frame_index.csv`) so the sky is
the ladder's sky rather than the two stray moonlit sessions sitting in the same folder.

In [ ]:
POOLS = {
    "bias":  dict(measured_type="bias"),
    "dark":  dict(measured_type="dark", exptime=60.0),
    "flat":  dict(measured_type="flat", exptime=3.0),
    "light": dict(measured_type="light", exptime=60.0),
}
LADDER_NIGHTS = ("2025-08-17", "2025-08-18", "2025-08-19")


def most_typical(pool):
    '''The frame nearest its pool's median, each column scaled by that pool's own
    std so no single large-valued column (`level`) decides the answer alone.'''
    x = pool[MEASURED].to_numpy(float)
    scale = np.where(x.std(0) > 0, x.std(0), 1.0)
    d = (((x - np.median(x, 0)) / scale) ** 2).sum(1)
    return pool.iloc[int(np.argmin(d))]


base = ok[(ok.gain == 252) & (ok.set_temp == -10)]
rows = []
for name, sel in POOLS.items():
    pool = base
    for col, val in sel.items():
        pool = pool[pool[col] == val]
    if name == "light":
        pool = pool[pool.date_obs.str[:10].isin(LADDER_NIGHTS)]
    print(f"{name:6s} pool of {len(pool):4d}")
    rows.append(most_typical(pool))

ex = pd.DataFrame(rows).set_index("measured_type")
ex[["exptime", "gain", "offset", "set_temp", "ccd_temp", "object", "date_obs"]]

In [ ]:
for t, p in ex.path.items():
    print(f"{t:6s} {pathlib.Path(p).name}")

## What the library actually looked at

A frame is 3840 x 2160 and `01` never read one whole (D35, and the sampling note in `fits.py`):
it read **six 32-row blocks** spread down the frame. Every number in the index comes from those
192 rows - 0.9% of the frame - and knowing that is half of knowing what the columns can and
cannot say.

Two properties of that geometry are doing real work. The blocks are **contiguous rows**, not
striped, because two of the features are spatial - a star is a blob, and striding would destroy
the blob. And they are **spread down the frame**, because vignetting and amp glow are
corner-weighted and a single central block would miss both.

In [ ]:
# `F.sample_blocks` returns stored values untouched -- converting inside the
# reader would destroy the evidence for the check that licenses converting.
# So the notebook converts here, once, and everything below is ADC counts.
blocks = {t: [ST.to_adc(b) for b in F.sample_blocks(p)[0]]
          for t, p in ex.path.items()}

fig, axes = plt.subplots(1, 4, figsize=(11, 3.4))
for ax, (t, bl) in zip(axes, blocks.items()):
    strip = np.vstack(bl).astype(float)
    lo, hi = np.percentile(strip, [1, 99.5])
    ax.imshow(strip, cmap="gray", vmin=lo, vmax=max(hi, lo + 1), aspect="auto")
    for k in range(1, len(bl)):                       # where the frame was skipped
        ax.axhline(k * bl[0].shape[0] - 0.5, color="tab:orange", lw=0.6)
    ax.set_title(f"{t}\n{strip.shape[0]}x{strip.shape[1]} px, "
                 f"{lo:.0f}-{hi:.0f} counts", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("the six sampled row-blocks, stacked; orange = a gap in the frame", y=1.02)
plt.tight_layout(); plt.show()

The flat is a smooth ramp, the light is stars on a sky, and **bias and dark are visually the same
thing** - which is the first useful lesson: no pixel statistic separates a 60 s dark from a 1 ms
bias on this sensor at this temperature. `classify` does not try. It settles bias from `exptime`,
a trusted capture setting, *before* looking at a pixel (D18).

## `level` and the per-plane `med_*` - where the frame sits

`level` is the mean of the four sub-plane medians, and each `med_R/G1/G2/B` is a median over
blocks of that plane's median. Medians throughout, and per CFA sub-plane throughout - never the
mosaic pooled, because on an OSC sensor the four planes sit at *different levels* under coloured
light, and pooling them would report a multi-modal mess as a single centre (D4).

In [ ]:
cols = ["level"] + [f"med_{p.lower()}" for p in S.PLANES]
lev = ex[cols].copy()
lev["B/G1"] = ex.med_b / ex.med_g1
lev["above pedestal"] = ex.level - 77.0       # gain-252 pedestal in ADC counts, from `01`
lev.round(3)

Three things this table is worth having in an index for:

- **The pedestal is visible and constant.** Bias and dark both sit at exactly 77 counts, all four
  planes. That is the offset the camera adds so noise cannot clip at zero, and having it as a
  column means any frame that has *drifted* off it is a `sort_values` away.
- **`B/G1` is a colour, and it says what the frame saw.** The flat's ratio is well below 1 - its
  light source is warm, which is a panel, not the sky. Bias and dark read exactly 1.000 because
  they saw nothing: a dark frame has no colour, and a "dark" whose planes *do* differ has a light
  leak. That check costs nothing, because the columns are already there.
- **`level` is the entire classifier.** Not just the flat detector: `level` minus the pedestal
  for the frame's gain is what "how much light arrived" means, and that one difference separates
  bias from dark from flat from light across all 15,090 frames (D50). Every other column below is
  description.

## `sigma` - and the most important caveat in the index

`sigma` is `1.4826 x MAD`, per plane, median across blocks, meaned over planes. It is in the
index as a *description* of a frame's per-plane scale - and because a dark whose MAD has left the
floor has something wrong with it. **It is not a noise measurement, and nothing may fit to it** -
the module docstring says so and D24 records why. Here is the why, in the pixels.

In [ ]:
plane = S.split(blocks["bias"][0])["G1"].astype(float)
med = np.median(plane)
dev = np.abs(plane - med)

print(f"bias G1 block: {plane.shape} pixels, median {med:.0f}")
print("distinct pixel values :", np.unique(plane)[:8], "...")
print("distinct |deviations| :", np.unique(dev)[:8], "...")
print(f"\nMAD = {np.median(dev):.0f}  ->  sigma = 1.4826 x {np.median(dev):.0f}"
      f" = {1.4826 * np.median(dev):.3f}")
print("the only sigmas this block could ever have returned:",
      np.round(1.4826 * np.array([0, 0.5, 1, 1.5, 2]), 2))

The ADC quantises to whole counts, so every deviation from the median is a whole number too, so
**the MAD - an order statistic of those deviations - can only land on that same lattice.** Its
finest nonzero step is 1.4826 x 1 = 1.4826 counts.

The index proves it at scale: the smallest nonzero `sigma` anywhere in 15,090 frames is exactly
1.4826, and the values just above it are integers times 1.4826 (taking a median across six blocks
and a mean across four planes halves the step a couple of times, but invents no resolution).

In [ ]:
s = np.sort(ok.sigma.unique())
print("smallest sigmas in the whole index:", np.round(s[:6], 3))
print("as multiples of 1.4826            :", np.round(s[1:6] / 1.4826, 4))
rungs = 5 * 1.4826            # a threshold in counts, not a stored-unit relic
print(f"\nframes in the first five rungs (sigma < {rungs:.2f} counts): "
      f"{(ok.sigma < rungs).mean():.1%}\n")
# The per-plane spreads are computed for every frame and stored for none: the
# index keeps their mean, as `sigma`.  Here they are, live, to show why.
per_plane = pd.DataFrame(
    {t: {p: 1.4826 * np.median(np.abs(pl - np.median(pl)))
         for p, pl in ((n, v.astype(float))
                       for n, v in S.split(blocks[t][0]).items())}
     for t in ex.index}).T[list(S.PLANES)]
per_plane.insert(0, "sigma (stored)", ex.sigma)
print(per_plane.round(2).to_string())

Read noise on this camera at gain 252 is of order a single ADC count, and the estimator's
first rung is 1.4826 counts: the quantity is smaller than the ruler's smallest division. So
the bias and the 60 s dark both report **1.4826** - one rung, the first one - and whatever
real difference is between them is invisible. That is not a discovery about the sensor. It is the estimator hitting
its floor, and it is exactly why a real noise estimator is build step 2.

The flat and the light report large sigmas, and those *are* meaningful as an ordering - but they
are dominated by scene structure inside the block, not by read noise either.


## The whole-frame summary - `mean`, `median`, `min`, `max`, `std`, `sampled_px`

Six columns that are **context, not evidence**. `classify` reads none of them, and neither does
`most_typical` above. They exist so a row can be eyeballed without opening the frame.

One of them is a trap worth walking into deliberately. `std` is pooled *across* the four CFA
planes, so on anything with colour in it, it is not measuring noise - it is measuring the
distance between the colours. `sigma` is the same frames' uncontaminated per-plane spread. The
two sit side by side in the index precisely so the gap can be seen.

In [ ]:
ratio = ok["std"] / ok["sigma"].where(lambda s: s > 0)

print(ex[SUMMARY].round(2).to_string(), "\n")
print("pooled std / per-plane sigma:")
print(ratio.groupby(ok.measured_type)
           .describe(percentiles=[.05, .5, .95])[["5%", "50%", "95%", "max"]]
           .round(1).to_string())

px = int(ex.sampled_px.iloc[0])
print(f"\nsampled_px = {px:,} of {3840 * 2160:,} = "
      f"{px / (3840 * 2160):.1%} of the frame")

**Where there is no colour, the two agree.** Bias and dark sit at 0.6-1.2: the pooled spread and
the per-plane spread are the same number, because all four planes saw the same nothing.

**Where there is colour, `std` is several times larger** - a median of 14.5x on flats and 10.3x
on lights, reaching 19x. None of that is noise. It is the R-to-G-to-B offset, and a noise fit fed
`std` would read the panel's colour temperature as read noise. This is `CLAUDE.md`'s mosaic rule
in one column: pooling across planes does not add a little error, it changes what is being
measured.

The ratio is therefore a **colour detector** in its own right. A "dark" whose ratio is 5 rather
than 1 saw light.

**And `sampled_px` is the denominator under every number in the row.** Six 32-row blocks is 8.9%
of the frame, so none of these will match a full-frame tool, and they are not meant to.

## `block_spread` - large-scale structure, in one number

`(max - min)` of the six per-block levels, divided by `level`. The blocks are spread down the
frame, so this is a crude **vertical gradient**: vignetting, amp glow, a sky gradient, a
satellite trail crossing one block.

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 3.1))
for t, bl in blocks.items():
    prof = [np.mean([np.median(pl) for pl in S.split(b).values()]) for b in bl]
    ax.plot(np.array(prof) / np.mean(prof), "o-",
            label=f"{t}  block_spread={ex.loc[t, 'block_spread']:.4f}")
ax.set_xlabel("block, top of frame -> bottom")
ax.set_ylabel("block level / frame level")
ax.legend(fontsize=7); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

The flat's value is its vignette profile - small, because these are six thin rows out of a smooth
bowl. The light's is several times larger, and it is sky gradient plus the target itself:
NGC 7000 is a large emission region and the blocks cross different parts of it. Bias and dark are
flat to the digit - **0.000000** - which is a genuine statement about this sensor: no measurable
amp glow at 60 s and gain 252.

Kept in the index because it is the cheapest available flag for "this frame has structure a
calibration frame should not have". A `block_spread` outlier among darks is a light leak or a
stray-light frame, findable without opening anything.

## The family that used to live here, and why it is gone

Earlier builds of this index carried four more columns - `tail_frac`, `clump_frac`, `clump_h`,
`clump_v`. They counted pixels more than five sigma above their plane's median and asked whether
those pixels **touched each other**, on the theory that a star is connected both ways and a hot
pixel is connected neither.

The theory was sound and the threshold was not. Five sigma meant five times *that frame's own*
MAD, so a bright sky raised the bar until no star cleared it: the brighter the sky, the more
star-blind the test became, and it failed hardest on the frames with the most signal. 484 real
lights were published as `dark`, 141 of them shot through trees and cloud - frames with no stars
in them at all, which no repair to a star test could ever have reached. Level above the pedestal
separates the same frames with no overlap and no reference to stars, so `classify` now reads one
number and these four columns are gone (D50).

`spatial.bright_pixels` is still there. It was always the right tool for a hot-pixel census; it
was the wrong tool for deciding what a frame *is*.

## `mult16_frac` and `sat_frac` - the two integrity columns

Neither describes the scene. Both describe whether the file can be trusted.

In [ ]:
print(ex[["mult16_frac", "sat_frac"]].to_string(), "\n")
print("mult16_frac across all readable frames:")
print(ok.mult16_frac.agg(["min", "mean", "max"]).to_string())
print(f"\nframes with any saturation at all: {(ok.sat_frac > 0).sum()}")
print(f"frames more than 1% saturated    : {(ok.sat_frac > 0.01).sum()}")

`mult16_frac = 1.000000` at min, mean and max, over every readable frame. That is the one
measurement of the rig `01` actually made, and it is what **licenses the project's one unit**:
if every stored value is an exact multiple of 16, then `stored / 16` is lossless and the whole
project can work in ADC counts, where header `EGAIN` applies directly with no factor to
remember. Note the order - the check runs on the *stored* values, before conversion, because
reading it off converted data would be circular.

It is a column rather than an assumption because the day a frame arrives with
`mult16_frac < 1` - a different camera, a processed file, a driver change - the index shows it,
and `to_adc` raises instead of the model quietly being wrong.

`sat_frac` is a **quality attribute, not a type** (D25, D27). A clipped frame is still a light;
it is just a light that cannot be used for photometry. Keeping the two ideas apart is what lets
`classify` say "bright and long => sky" without also having to decide the frame is unusable.

## The whole decision, replayed

`classify` reads in a fixed order, and the order is the argument. Exposure settles bias before
any pixel is consulted; then a frame three orders of magnitude above the pedestal at an exposure
of seconds is a panel and at an exposure of minutes is sky; then what is left is dark if it sits
on its pedestal and light if it does not.

**`light` is the fallback, and that is a safety property.** Every frame the earlier branches do
not claim lands there, because a light wrongly called `dark` goes into a calibration master and
is subtracted from every science frame, while a dark wrongly called `light` is thrown out by
registration. The old classifier fell through to `dark`, which is how 484 lights ended up in the
wrong bucket.

The pedestal is passed in rather than hard-coded: `01` measures it from bias frames, which are
selected by exposure alone, so nothing here argues in a circle.

In [ ]:
bias = ok[ok.exptime <= ST.BIAS_MAX_EXPTIME]
PEDESTAL = bias.groupby("gain")["level"].median()
print(f"pedestal from {len(bias)} bias frames, by gain: "
      f"{ {g: float(v) for g, v in PEDESTAL.items()} }\n")

FLAT_CUT = ST.FLAT_MIN_LEVEL * ST.ADC_FULL_SCALE


def trace(row):
    e, ped = float(row.exptime), PEDESTAL[row.gain]
    above = row.level - ped
    if e <= ST.BIAS_MAX_EXPTIME:
        return f"exptime {e} <= {ST.BIAS_MAX_EXPTIME} s  (no pixel consulted)"
    if above >= FLAT_CUT and e <= ST.FLAT_MAX_EXPTIME:
        return (f"{above:7.1f} counts above pedestal {ped:.0f} >= {FLAT_CUT:.0f}"
                f", at {e} s <= {ST.FLAT_MAX_EXPTIME} s")
    if above <= ST.DARK_MAX_ABOVE_PEDESTAL:
        return (f"{above:7.2f} counts above pedestal {ped:.0f} "
                f"<= {ST.DARK_MAX_ABOVE_PEDESTAL}")
    return (f"{above:7.1f} counts above pedestal {ped:.0f} "
            f"> {ST.DARK_MAX_ABOVE_PEDESTAL}  (light is the fallback)")


for t, row in ex.iterrows():
    verdict = ST.classify(row[MEASURED].to_dict(), row.exptime,
                          PEDESTAL.get(row.gain))
    flag = "" if verdict == t else "   <-- disagrees with the pool it came from"
    print(f"{t:6s} -> {verdict:6s}  because {trace(row)}{flag}")

## What the columns are for

| column | what it measures | why the archive keeps it |
|---|---|---|
| `level` | where the frame sits, meaned over the four CFA planes | **the one column `classify` reads**: `level` minus the pedestal for the gain is how much light arrived (D50) |
| `med_*` | the same, per plane | pedestal drift; plane ratios are a colour, and a colourless "dark" that has one has a light leak |
| `sigma` | MAD scale, computed per plane and stored as their mean (D24) | quantised at 1.4826 counts, so never a noise number; of 2,497 zero-light frames exactly one has left the floor, and it is also the one with structure and colour it should not have |
| `block_spread` | vertical structure across the frame | vignetting, sky gradient, amp glow; an outlier among darks is stray light |
| `mult16_frac` | fraction of values that are multiples of 16 | proves the 12-bit-in-16-bit container, on every single frame |
| `sat_frac` | fraction at full scale | quality flag, orthogonal to frame type (D25, D27) |
| `mean`, `median`, `min`, `max`, `std` | whole-frame summary, pooled across planes | orientation only, read by nothing; `std` is channel balance, not noise |
| `sampled_px` | how many pixels the row was computed from | the honest denominator - 8.9% of the frame, so no number here matches a full-frame tool |

**The shape of the whole thing.** One classifier number, four more of description and integrity,
four per-plane medians, six of summary, six 32-row blocks, four sub-planes, and no debayering
anywhere.
That is enough to type 15,090 frames from their pixels and to make the corpus queryable - and it
is deliberately *not* enough to measure anything. The sensor constants come later, from frames
shot to a protocol into `data/`, with a real estimator.

**One column carries the whole verdict, and that is on purpose.** A classifier reading nine
features looks more careful than one reading one, and the old one was not: it was reading eight
numbers that could not tell it what it needed and one that could. Deleting the eight did not make
it weaker.

---

**Next.** Nothing in `results/` changed here - `01` remains the only writer of
`frame_index.csv`. The question `01` used to leave open, which of the 684 `light`-labelled,
`dark`-measuring frames were mislabelled darks and which were missed lights, is closed: 200 were
darks and 484 were lights, and the classifier that got them wrong has been replaced by one that
does not (D50).